In [1]:
!pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm --no-index --find-links=/kaggle/input/datasets/ziyankhanpathan/dicom-new/dicom-wheels


Looking in links: /kaggle/input/datasets/ziyankhanpathan/dicom-new/dicom-wheels
ERROR: Could not find a version that satisfies the requirement pylibjpeg (from versions: none)
ERROR: No matching distribution found for pylibjpeg


In [2]:
from __future__ import annotations

import gc
import glob
import json
import math
import os
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydicom.pixel_data_handlers.util import apply_modality_lut

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

T0 = time.time()
TIME_BUDGET = 8.5 * 3600  # 8.5 hours
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Targets (submission order) ─────────────────────────────────────────────────
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_TARGETS = len(TARGETS)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVS = []
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        try:
            _cc = torch.cuda.get_device_capability(_i)
            _dev = torch.device(f"cuda:{_i}")
            _p = nn.Conv2d(3, 4, 3, padding=1).eval().to(_dev)
            with torch.inference_mode():
                _p(torch.zeros(1, 3, 16, 16, device=_dev))
            DEVS.append(_dev)
            print(f"cuda:{_i} OK — {torch.cuda.get_device_name(_i)} sm_{_cc[0]}{_cc[1]}")
            del _p
        except Exception as _e:
            print(f"cuda:{_i} SKIP ({_e})")
if not DEVS:
    DEVS = [torch.device("cpu")]
DEV = DEVS[0]
print(f"Primary device: {DEV}")

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

# %% [markdown]
# ## Path Discovery
# Finds the competition data and all weight packages automatically.

# %% [code] — Path discovery
def _find_dir(*candidates):
    for c in candidates:
        if os.path.isdir(c):
            return Path(c)
    return None

COMP_ROOT = _find_dir(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection",
    "/kaggle/input/rsna-knee-abnormality-detection",
)
if COMP_ROOT is None:
    raise RuntimeError("Competition data not found. Attach rsna-knee-abnormality-detection.")

DINO_FOLD_DIR = _find_dir(
    "/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-weights",
    "/kaggle/input/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-weights",
)

RAD_HEADS_DIR = _find_dir(
    "/kaggle/input/datasets/mattiaangeli/rsna-knee-radimagenet-foldsv1-heads",
    "/kaggle/input/rsna-knee-radimagenet-foldsv1-heads",
)

RAPTOR_DIR = _find_dir(
    "/kaggle/input/datasets/dreaddevelopment/raptor-knee-widedense",
    "/kaggle/input/raptor-knee-widedense",
)
# If Raptor dir wasn't found by exact path, find any directory containing the raptor weights
if RAPTOR_DIR is None:
    for p in Path("/kaggle/input").rglob("raptor_ft_coatnet_v4_full.pt"):
        RAPTOR_DIR = p.parent
        break

WORKING = Path("/kaggle/working")
WORKING.mkdir(exist_ok=True)

log(f"Competition  : {COMP_ROOT}")
log(f"DINO Folds   : {DINO_FOLD_DIR}")
log(f"Rad Heads    : {RAD_HEADS_DIR}")
log(f"Raptor dir   : {RAPTOR_DIR}")

TEST_CSV   = COMP_ROOT / "test.csv"
TRAIN_CSV  = COMP_ROOT / "train.csv"
_TEST_SER_NAME = "test_series" if (COMP_ROOT / "test_series").exists() else "train_series"
TEST_SER   = COMP_ROOT / _TEST_SER_NAME
TEST_SER_CSV = COMP_ROOT / "test_series.csv"

test_df  = pd.read_csv(TEST_CSV)
test_ser = pd.read_csv(TEST_SER_CSV)
sample   = pd.read_csv(COMP_ROOT / "sample_submission.csv")
log(f"Test studies: {len(test_df)} | Test series: {len(test_ser)}")

# %% [markdown]
# ## DICOM Metadata Parsing
#
# We parse every series header to recover:
# - Anatomical plane (Sagittal / Coronal / Axial)
# - Fat suppression flag
# - Contrast weighting (T1 / T2 / PD)
# - Pixel spacing
# - Laterality
#
# These metadata are used to (a) assign series to slots and (b) compute
# per-finding slot-coverage scores for metadata-conditioned blending.

# %% [code] — DICOM header reading and series annotation
_SEP       = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bfatsup\b")
_T1_RX     = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX     = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX     = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}

HDR_TAGS = [
    "SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
    "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing",
    "Rows", "Columns", "ImagePositionPatient", "ImageOrientationPatient",
]

def _probe_series(item):
    """Read one representative DICOM header from a series directory."""
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study,
           "SeriesInstanceUID": series, "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(
            os.path.join(path, files[len(files) // 2]),
            stop_before_pixels=True, force=True,
        )
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as e:
        row["err"] = str(e)[:80]
    return row

def _walk_split(split_name):
    """Collect all series directories under a split."""
    base = COMP_ROOT / split_name
    items = []
    if not base.is_dir():
        return []
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split_name, study.name, series.name, series.path))
    return items

def read_headers(split_name, n_threads=12):
    items = _walk_split(split_name)
    log(f"Probing {len(items)} series in {split_name} ...")
    with ThreadPoolExecutor(max_workers=n_threads) as pool:
        rows = list(pool.map(_probe_series, items))
    return pd.DataFrame(rows)

def annotate_series(df):
    """Add fat-suppression, fluid-sensitivity, contrast-weighting and pixel-spacing columns."""
    desc = df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna("")
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)
    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"],       errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
    df["weight"] = np.where(
        t1 & ~t2 & ~pdw, "T1",
        np.where(t2 & ~pdw, "T2",
        np.where(pdw, "PD",
        np.where(gre, "GRE",
        np.where(tr < 800, "T1",
        np.where(te > 60, "T2",
        np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = df["weight"].isin(["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce",
    )
    return df

log("Reading test series headers...")
hdr = annotate_series(read_headers(_TEST_SER_NAME))
log(f"Headers: {len(hdr)} series annotated")

# %% [markdown]
# ## Slot Assignment and Laterality Detection
#
# Six named slots per study. Laterality is detected from image geometry
# (median x-coordinate of ImagePositionPatient across all slices in a study).

# %% [code] — Slot scheme and slot assignment
# 6-slot recovered scheme  ── name, plane, fluid, fat-saturated
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal",  True,  True),
    ("COR_FLUID_FS",   "Coronal",   True,  True),
    ("AX_FLUID_FS",    "Axial",     True,  True),
    ("SAG_FLUID_NOFS", "Sagittal",  True,  False),
    ("COR_T1",         "Coronal",   False, False),
    ("SAG_T1",         "Sagittal",  False, False),
]
N_SLOT      = len(SLOTS)
SLOT_NAMES  = [s[0] for s in SLOTS]

# Map of plane names from the competition series CSV
# The column may be named "Anatomical_Plane" or "anatomical_plane" depending on version
_plane_col = "Anatomical_Plane" if "Anatomical_Plane" in test_ser.columns else (
    "anatomical_plane" if "anatomical_plane" in test_ser.columns else None
)
if _plane_col:
    PLANE_MAP = dict(zip(test_ser["SeriesInstanceUID"], test_ser[_plane_col]))
else:
    log("WARNING: No Anatomical_Plane column in test_series.csv — slot assignment may fail.")
    PLANE_MAP = {}

def detect_laterality(h):
    """Infer left/right from explicit DICOM tags with fallback to median x-coordinate of image positions."""
    out = {}
    n_tag, n_geo, n_none, n_disagree = 0, 0, 0, 0
    for study, g in h.groupby("StudyInstanceUID"):
        # 1. Try explicit tags first
        v = [str(x).strip().upper() for x in g.get("Laterality", pd.Series(dtype=str)).dropna()]
        if "ImageLaterality" in g.columns:
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        tag_side = v[0] if v else None
        
        # 2. Try geometry
        geo_side = None
        xs = []
        for r in g.itertuples():
            ipp = getattr(r, "ImagePositionPatient", None)
            if isinstance(ipp, str):
                try:
                    vals = [float(val) for val in ipp.split("|")]
                    if len(vals) >= 1:
                        xs.append(vals[0])
                except Exception:
                    pass
        if xs:
            m = float(np.median(xs))
            geo_side = "R" if m < -20 else ("L" if m > 20 else None)
            
        # 3. Consensus
        if tag_side is not None:
            n_tag += 1
            if geo_side is not None and tag_side != geo_side:
                n_disagree += 1
            out[study] = tag_side
        else:
            if geo_side is not None:
                n_geo += 1
            else:
                n_none += 1
            out[study] = geo_side

    log(f"Laterality: {n_tag} from tags, {n_geo} from geometry, {n_none} unknown; disagreed on {n_disagree}")
    return out

lat_map = detect_laterality(hdr)

def pick_slots(hdr_df, plane_map):
    """Select best series per slot per study with strict uniqueness and T1 fluid exclusion."""
    hdr_df = hdr_df.copy()
    hdr_df["plane"] = hdr_df["SeriesInstanceUID"].map(plane_map).fillna(
        hdr_df["SeriesInstanceUID"].map(dict(zip(hdr_df["SeriesInstanceUID"], hdr_df.get("Anatomical_Plane", pd.Series(dtype=str)))))
    )
    out = {}
    for study, g in hdr_df.groupby("StudyInstanceUID"):
        chosen = {}
        used_series = set()
        for name, plane, fluid, fs in SLOTS:
            g_avail = g[~g["SeriesInstanceUID"].isin(used_series)]
            sel = (g_avail["plane"] == plane) & (g_avail["fatsat"] == fs)
            if fluid:
                sel &= g_avail["fluid"] == True
            else:
                sel &= g_avail["fluid"] == False
            cand = g_avail[sel]
            if len(cand) == 0 and not fluid:
                cand = g_avail[(g_avail["plane"] == plane) & ~g_avail["fatsat"]]
            if len(cand):
                best = cand.sort_values("n_slices", ascending=False).iloc[0].to_dict()
                chosen[name] = best
                used_series.add(best["SeriesInstanceUID"])
        out[study] = chosen
    return out

from scipy.stats import spearmanr

def check_backbone_keys(model, state_dict, model_name="Model", max_missing_ratio=0.30):
    """Strictly verifies backbone state dict key alignment before inference."""
    total_keys = len(model.state_dict())
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    missing_ratio = len(missing) / max(total_keys, 1)
    log(f"[{model_name}] State dict match: {len(missing)} missing, {len(unexpected)} unexpected out of {total_keys} keys.")
    if missing_ratio > max_missing_ratio:
        raise RuntimeError(f"CRITICAL: {model_name} key mismatch! {len(missing)}/{total_keys} ({missing_ratio:.1%}) keys missing.")
    return len(missing), len(unexpected)

def log_pairwise_spearman(member_preds_dict, targets=TARGETS):
    """Computes and logs pairwise Spearman rank correlations between ensemble members."""
    members = [k for k, v in member_preds_dict.items() if v is not None]
    if len(members) < 2:
        log("Pairwise Spearman check: Fewer than 2 active members.")
        return
    log(f"=== Pairwise Spearman Correlation Check ({len(members)} Members) ===")
    for c, target in enumerate(targets):
        pair_rhos = []
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                m1, m2 = members[i], members[j]
                p1, p2 = member_preds_dict[m1][:, c], member_preds_dict[m2][:, c]
                if np.std(p1) > 1e-8 and np.std(p2) > 1e-8:
                    rho, _ = spearmanr(p1, p2)
                else:
                    rho = 0.0
                pair_rhos.append(f"{m1} vs {m2}: {rho:.3f}")
                if rho < 0.20:
                    log(f"  [WARNING] Low agreement on '{target}' between {m1} & {m2} (Spearman = {rho:.3f})")
        log(f"  Target '{target}': {', '.join(pair_rhos[:2])} ...")

def audit_dicom_geometry_and_slots(hdr_df, slot_map):
    """Audits slice ordering and slot duplication rates."""
    if hdr_df is None or len(hdr_df) == 0:
        return
    dup_count = 0
    total_studies = len(slot_map)
    for study, chosen in slot_map.items():
        uids = [v["SeriesInstanceUID"] for v in chosen.values() if isinstance(v, dict) and "SeriesInstanceUID" in v]
        if len(uids) > len(set(uids)):
            dup_count += 1
    dup_rate = dup_count / max(total_studies, 1)
    log(f"Slot duplication audit: {dup_count}/{total_studies} studies ({dup_rate:.1%}) have duplicate series assigned to multiple slots.")

slot_map = pick_slots(hdr, PLANE_MAP)
n_covered = sum(1 for v in slot_map.values() if len(v) > 0)
log(f"Slot map: {n_covered}/{len(test_df)} studies have at least one slot")
audit_dicom_geometry_and_slots(hdr, slot_map)

# %% [markdown]
# ## Metadata Features and Slot Coverage
#
# For each study, compute a slot-presence vector and then per-finding
# slot-coverage scores. These scores tell us how reliable each arm's
# prediction is for each finding on each specific study.
#
# The slot-prior table comes from the notebook analysis — each finding
# is best visible in certain slots.

# %% [code] — Slot coverage and metadata features
# Slot-prior table: for each finding, which slot indices matter
SLOT_PRIOR = {
    "ACL":              [0, 3, 5],   # sagittal slots
    "MCL":              [1, 4],      # coronal slots
    "Medial Meniscus":  [0, 1, 3, 4],
    "Lateral Meniscus": [0, 1, 3, 4],
    "Medial OA":        [1, 4, 5],
    "Lateral OA":       [1, 4, 5],
    "PF OA":            [0, 2, 5],
    "Effusion":         [0, 2],
    "Synovitis":        [0, 2],
    "Baker's":          [0],
    "Contusion":        [0, 1, 2],
    "Fracture":         [0, 1, 2, 4, 5],
}

def compute_coverage(slot_map, test_ids):
    """
    Returns:
        slot_presence : (N, N_SLOT) float32 — 1 if slot is present for study i
        finding_coverage : (N, N_TARGETS) float32 — fraction of important slots present
                           for each finding for each study.
    """
    N = len(test_ids)
    slot_presence  = np.zeros((N, N_SLOT), dtype=np.float32)
    finding_coverage = np.zeros((N, N_TARGETS), dtype=np.float32)

    for i, study in enumerate(test_ids):
        chosen = slot_map.get(study, {})
        for j, name in enumerate(SLOT_NAMES):
            if name in chosen:
                slot_presence[i, j] = 1.0

        for k, target in enumerate(TARGETS):
            important = SLOT_PRIOR.get(target, list(range(N_SLOT)))
            present   = sum(slot_presence[i, j] for j in important)
            finding_coverage[i, k] = present / max(len(important), 1)

    log(f"Coverage: mean slot fill {slot_presence.mean():.2f}, "
        f"mean finding coverage {finding_coverage.mean():.2f}")
    return slot_presence, finding_coverage

TEST_IDS = test_df["StudyInstanceUID"].astype(str).tolist()
slot_presence, finding_coverage = compute_coverage(slot_map, TEST_IDS)

# %% [markdown]
# ## Utility: Rank Normalisation and Combining Arms
#
# All combining is done via percentile rank averaging.
# `rank_pct(x)` converts column-wise scores into [0, 1] percentile ranks.
# `fold_rank_aggregate(preds_list)` does fold-level aggregation:
#   1. Average predictions within fold
#   2. Rank those fold means
#   3. Average the fold ranks
# This is the main anti-overfitting mechanism.

# %% [code] — Rank utilities
def rank_pct(x: np.ndarray) -> np.ndarray:
    """Per-column percentile rank. Input: (N, C). Output: (N, C) in [0, 1]."""
    out = np.empty_like(x, dtype=np.float64)
    for c in range(x.shape[1]):
        col = x[:, c]
        order = col.argsort().argsort().astype(np.float64)
        out[:, c] = order / max(len(col) - 1, 1)
    return out.astype(np.float32)

def fold_rank_aggregate(fold_preds: list[np.ndarray]) -> np.ndarray:
    """
    fold_preds: list of (N, N_TARGETS) arrays — one per fold.
    Returns: (N, N_TARGETS) rank-aggregated predictions.

    Fold-level aggregation prevents any one fold family from dominating.
    """
    fold_ranks = [rank_pct(p) for p in fold_preds]
    return np.mean(fold_ranks, axis=0).astype(np.float32)

def weighted_rank_mean(arm_preds: list[np.ndarray], weights: list[float]) -> np.ndarray:
    """
    arm_preds: list of (N, N_TARGETS) prediction arrays (should already be rank-normalised)
    weights: list of floats, same length as arm_preds
    """
    w = np.array(weights, dtype=np.float64)
    w = w / w.sum()
    out = np.zeros_like(arm_preds[0], dtype=np.float64)
    for arm, wi in zip(arm_preds, w):
        out += wi * arm.astype(np.float64)
    return out.astype(np.float32)

# Per-finding pooling strategy over windows
# Focal findings: max pooling over windows (one slice showing it is enough)
# Diffuse findings: mean pooling
FOCAL_FINDINGS = {"ACL", "MCL", "Baker's", "Fracture", "Contusion",
                  "Medial Meniscus", "Lateral Meniscus"}

def apply_window_pool(window_probs: np.ndarray, target: str) -> np.ndarray:
    """
    window_probs: (n_windows, N, N_TARGETS)
    Returns: (N, N_TARGETS)
    """
    if target in {"ACL", "MCL"}:
        # Robust pooling for specific focal targets: average top 2 windows to reduce artifact spikes
        if window_probs.shape[0] >= 2:
            return np.sort(window_probs, axis=0)[-2:].mean(axis=0)
        return window_probs.max(axis=0)
    elif target in FOCAL_FINDINGS:
        return window_probs.max(axis=0)
    return window_probs.mean(axis=0)

# %% [markdown]
# ## Metadata-Conditioned Blending
#
# Standard approach: blend arms at fixed weights.
# Our approach: multiply each arm's per-finding weight by the slot-coverage
# score for that finding for that study.
#
# Studies where important slots are missing get reduced confidence in those
# findings. The scores are renormalized per finding so the final blend
# still sums to 1 across arms.

# %% [code] — Metadata-conditioned blending
def rank_pct(x: np.ndarray) -> np.ndarray:
    """Percentile rank per column in [0, 1]. Preserves strict ordering without flat ties."""
    n = len(x)
    if n <= 1:
        return np.full_like(x, 0.5, dtype=np.float32)
    from scipy.stats import rankdata
    ranks = np.zeros_like(x, dtype=np.float32)
    for c in range(x.shape[1]):
        r = rankdata(x[:, c], method="average")
        ranks[:, c] = (r - 0.5) / float(n)
    return ranks

def apply_hierarchical_consistency(preds: np.ndarray, targets: list = TARGETS, train_gt_df=None) -> np.ndarray:
    """
    Applies continuous, strictly monotonic probabilistic calibration across the 12 RSNA target heads.
    Uses soft multiplicative, copula, and logistic constraints rather than hard clipping to eliminate rank ties.
    Optionally calibrates threshold parameters dynamically if ground truth labels (train_gt_df) are provided.
    """
    out = preds.copy()
    idx = {t: i for i, t in enumerate(targets)}

    fluid_thresh = 0.15
    contusion_thresh = 0.70

    if train_gt_df is not None and len(train_gt_df) > 0:
        try:
            # GT columns in train.csv are binary indicators (0 or 1).
            # Compute positive label prevalence rate to calibrate confidence thresholds.
            if "Effusion" in train_gt_df.columns:
                eff_prev = float(train_gt_df["Effusion"].mean())
                fluid_thresh = max(0.10, min(0.30, 1.0 - eff_prev))
            if "Contusion" in train_gt_df.columns:
                cont_prev = float(train_gt_df["Contusion"].mean())
                contusion_thresh = max(0.60, min(0.85, 1.0 - cont_prev))
        except Exception as e:
            log(f"Dynamic GT parameter calibration fallback: {e}")

    # 1. Soft Joint Cavity Fluid & Posterior Cyst Calibration (Effusion/Synovitis -> Baker's)
    if "Effusion" in idx and "Synovitis" in idx and "Baker's" in idx:
        eff_idx, syn_idx, bak_idx = idx["Effusion"], idx["Synovitis"], idx["Baker's"]
        fluid_max = np.maximum(out[:, eff_idx], out[:, syn_idx])
        gate = 0.20 + 0.80 / (1.0 + np.exp(-10.0 * (fluid_max - fluid_thresh)))
        out[:, bak_idx] = out[:, bak_idx] * gate

    # 2. Medial & Lateral Compartment Soft Coupling (Meniscus <-> OA)
    if "Medial Meniscus" in idx and "Medial OA" in idx:
        med_m_idx, med_oa_idx = idx["Medial Meniscus"], idx["Medial OA"]
        medial_comp = 0.5 * (out[:, med_m_idx] + out[:, med_oa_idx])
        out[:, med_m_idx] = 0.90 * out[:, med_m_idx] + 0.10 * medial_comp
        out[:, med_oa_idx] = 0.90 * out[:, med_oa_idx] + 0.10 * medial_comp

    if "Lateral Meniscus" in idx and "Lateral OA" in idx:
        lat_m_idx, lat_oa_idx = idx["Lateral Meniscus"], idx["Lateral OA"]
        lateral_comp = 0.5 * (out[:, lat_m_idx] + out[:, lat_oa_idx])
        out[:, lat_m_idx] = 0.90 * out[:, lat_m_idx] + 0.10 * lateral_comp
        out[:, lat_oa_idx] = 0.90 * out[:, lat_oa_idx] + 0.10 * lateral_comp

    # 3. Trauma Coupling (Contusion -> Fracture)
    if "Fracture" in idx and "Contusion" in idx:
        frac_idx, cont_idx = idx["Fracture"], idx["Contusion"]
        boost = 0.15 * np.maximum(0.0, out[:, cont_idx] - contusion_thresh)
        out[:, frac_idx] = np.maximum(out[:, frac_idx], out[:, frac_idx] + boost)

    return out

FINDING_ARM_PRIORS = {
    "ACL":              {"raptor": 0.35, "dinov3": 0.52, "radimagenet": 0.13},
    "MCL":              {"raptor": 0.35, "dinov3": 0.52, "radimagenet": 0.13},
    "Medial Meniscus":  {"raptor": 0.36, "dinov3": 0.50, "radimagenet": 0.14},
    "Lateral Meniscus": {"raptor": 0.36, "dinov3": 0.50, "radimagenet": 0.14},
    "Medial OA":        {"raptor": 0.48, "dinov3": 0.40, "radimagenet": 0.12},
    "Lateral OA":       {"raptor": 0.48, "dinov3": 0.40, "radimagenet": 0.12},
    "PF OA":            {"raptor": 0.48, "dinov3": 0.40, "radimagenet": 0.12},
    "Effusion":         {"raptor": 0.32, "dinov3": 0.43, "radimagenet": 0.25},
    "Synovitis":        {"raptor": 0.32, "dinov3": 0.43, "radimagenet": 0.25},
    "Baker's":         {"raptor": 0.45, "dinov3": 0.55, "radimagenet": 0.00},
    "Contusion":        {"raptor": 0.45, "dinov3": 0.45, "radimagenet": 0.10},
    "Fracture":         {"raptor": 0.50, "dinov3": 0.50, "radimagenet": 0.00},
}

def metadata_conditioned_blend(
    arm_preds:    list[np.ndarray],
    base_weights: list[float],
    arm_coverages: list[np.ndarray] = None,
    alpha: float = 0.4,
    arm_families: list[str] = None,
) -> np.ndarray:
    """
    Masked Dynamic Blending (Probability Logit + Rank) with Per-Arm Coverage Modulation.
    Uses NaN for invalid/missing studies to avoid ranking 0.5 fallbacks.
    """
    N, C = arm_preds[0].shape
    n_arms = len(arm_preds)

    if arm_families is not None and len(arm_families) == n_arms:
        base_w_matrix = np.zeros((n_arms, C), dtype=np.float64)
        for a, fam in enumerate(arm_families):
            fam_key = fam.lower()
            if "raptor" in fam_key or "coatnet" in fam_key: key = "raptor"
            elif "dino" in fam_key: key = "dinov3"
            elif "rad" in fam_key: key = "radimagenet"
            else: key = "raptor"

            fam_count = sum(1 for f in arm_families if key in f.lower() or (key=="raptor" and "coatnet" in f.lower()))
            if fam_count == 0: fam_count = 1

            for c, target in enumerate(TARGETS):
                priors = FINDING_ARM_PRIORS.get(target, {"raptor": 0.40, "dinov3": 0.45, "radimagenet": 0.15})
                base_w_matrix[a, c] = priors.get(key, 0.33) / fam_count
    else:
        base_w = np.array(base_weights, dtype=np.float64)
        base_w_matrix = np.tile(base_w[:, None], (1, C))

    eff_weights = np.empty((n_arms, N, C), dtype=np.float64)
    for a in range(n_arms):
        cov = arm_coverages[a] if (arm_coverages is not None and a < len(arm_coverages)) else np.ones((N, C), dtype=np.float64)
        modifier = 1.0 + alpha * (cov - 0.5)
        eff_weights[a] = base_w_matrix[a][None, :] * modifier

        nan_mask = np.isnan(arm_preds[a])
        eff_weights[a][nan_mask] = 0.0

    weight_sum = eff_weights.sum(axis=0, keepdims=True)
    valid_weight_mask = weight_sum > 1e-8
    eff_weights = np.where(valid_weight_mask, eff_weights / np.maximum(weight_sum, 1e-8), 0.0)

    rank_arms = [rank_pct(np.nan_to_num(a, nan=0.5)) for a in arm_preds]
    rank_blend = np.zeros((N, C), dtype=np.float64)
    for a in range(n_arms):
        rank_blend += eff_weights[a] * rank_arms[a].astype(np.float64)

    logit_blend = np.zeros((N, C), dtype=np.float64)
    for a in range(n_arms):
        p_clamped = np.clip(np.nan_to_num(arm_preds[a], nan=0.5), 1e-6, 1.0 - 1e-6)
        logits = np.log(p_clamped / (1.0 - p_clamped))
        logit_blend += eff_weights[a] * logits

    prob_blend = 1.0 / (1.0 + np.exp(-logit_blend))
    prob_rank = rank_pct(prob_blend)

    out = 0.50 * rank_blend + 0.50 * prob_rank
    return out.astype(np.float32)

CROP_MM   = 130.0
CACHE_IMG = 336
GROUP     = 3    # slices per slot in the DINOv3/Rad cache
SLICE_BAND = (0.2, 0.8)

def _order_by_geometry(files, d):
    """Order DICOM files by projection onto the slice normal, with dominant axis fallback."""
    rows = []
    for f in files:
        k, ipp_coords = None, None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImageOrientationPatient", "ImagePositionPatient", "InstanceNumber"])
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient",   None)
            if iop is not None and ipp is not None:
                iop_np = np.array([float(x) for x in iop])
                ipp_np = np.array([float(x) for x in ipp])
                n = np.cross(iop_np[:3], iop_np[3:])
                k = float(np.dot(ipp_np, n))
                ipp_coords = ipp_np
            elif ipp is not None:
                ipp_coords = np.array([float(x) for x in ipp])
        except Exception:
            pass
        
        if k is None:
            try:
                k_inst = float(getattr(ds, "InstanceNumber", 0) or 0)
            except Exception:
                k_inst = None
        else:
            k_inst = None
            
        rows.append((f, k, ipp_coords, k_inst))

    # 1. Try standard cross product first
    if sum([1 for r in rows if r[1] is not None]) == len(rows):
        rows.sort(key=lambda x: x[1])
        return [r[0] for r in rows]

    # 2. Try dominant axis fallback
    placed = [r for r in rows if r[2] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[2] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: float(r[2][axis]) if r[2] is not None else spare)
        return [r[0] for r in rows]

    # 3. Try instance number fallback
    if sum([1 for r in rows if r[3] is not None]) >= need:
        rows.sort(key=lambda r: r[3] if r[3] is not None else float("inf"))
        return [r[0] for r in rows]

    # 4. Give up and sort alphabetically
    rows.sort(key=lambda r: r[0])
    return [r[0] for r in rows]

def _read_slot(rec, n_slice=GROUP, out_size=CACHE_IMG, dynamic_saliency=True):
    """Decode one slot into a (n_slice, H, W) uint8 tensor with dynamic feature-energy slice selection."""
    files_raw = rec.get("files", [])
    d         = rec["dir"]
    px        = rec.get("px")

    # Try to order by geometry
    try:
        files = _order_by_geometry(files_raw, d)
    except Exception:
        files = files_raw

    n = len(files)
    if n == 0:
        return None

    lo_i = int(SLICE_BAND[0] * (n - 1))
    hi_i = int(SLICE_BAND[1] * (n - 1))

    # Fast strided dynamic contrast energy slice selection across band
    if dynamic_saliency and (hi_i - lo_i + 1) >= n_slice * 2:
        # Strided candidate sampling to minimize I/O overhead
        stride = max(1, (hi_i - lo_i + 1) // 12)
        cand_indices = list(range(lo_i, hi_i + 1, stride))
        energies = []
        for ci in cand_indices:
            try:
                ds_cand = pydicom.dcmread(os.path.join(d, files[ci]), force=True, stop_before_pixels=False)
                arr_c = apply_modality_lut(ds_cand.pixel_array, ds_cand).astype(np.float32)
                std_e = np.std(arr_c)
                energies.append(std_e)
            except Exception:
                energies.append(0.0)
        
        energies = np.array(energies)
        if energies.max() > 0:
            top_sorted = np.argsort(energies)[::-1]
            selected = []
            for ts in top_sorted:
                if all(abs(ts - s) >= max(1, len(cand_indices) // (n_slice * 2)) for s in selected):
                    selected.append(ts)
                if len(selected) == n_slice:
                    break
            selected.sort()
            idx = np.array([cand_indices[s] for s in selected])
        else:
            idx = np.unique(np.linspace(lo_i, hi_i, n_slice).astype(int))
    else:
        idx = np.unique(np.linspace(lo_i, hi_i, n_slice).astype(int))

    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    idx = idx[:n_slice]

    planes = []
    for i in idx:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a  = apply_modality_lut(ds.pixel_array, ds).astype(np.float32)
            if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
                a = a.max() - a
        except Exception:
            planes.append(None)
            continue
        planes.append(a)

    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        return None
    for k, p in enumerate(planes):
        if p is None:
            planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)  # (n_slice, H, W)

    # mm-based crop
    try:
        px_val = float(px) if px is not None else 0.0
    except (TypeError, ValueError):
        px_val = 0.0
    if px_val > 0 and np.isfinite(px_val):
        want = int(round(CROP_MM / px_val))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half   = want // 2
            vol    = vol[:, max(0, cy - half):cy + half,
                            max(0, cx - half):cx + half]

    # Intensity normalisation
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    # Resize
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)  # (1, n, H, W)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

def _flip_laterality(img, plane, lat):
    """Flip right knees so all studies face the same anatomical direction."""
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

def build_6slot_cache(test_ids, slot_map, lat_map):
    """Build the shared (N, N_SLOT, GROUP, H, W) uint8 cache for DINOv3 and RadImageNet."""
    N = len(test_ids)
    cache = np.zeros((N, N_SLOT, GROUP, CACHE_IMG, CACHE_IMG), dtype=np.uint8)
    mask  = np.zeros((N, N_SLOT), dtype=np.float32)

    log(f"Building 6-slot cache: {cache.shape}, {cache.nbytes / 1e9:.1f} GB")

    for i, study in enumerate(test_ids):
        chosen = slot_map.get(study, {})
        lat    = lat_map.get(study)
        for j, (name, plane, _, _) in enumerate(SLOTS):
            if name not in chosen:
                continue
            rec = chosen[name]
            img = _read_slot(rec, GROUP, CACHE_IMG)
            if img is None:
                continue
            img = _flip_laterality(img, plane, lat)
            cache[i, j] = img.numpy()
            mask[i, j]  = 1.0

        if (i + 1) % 100 == 0 or i + 1 == N:
            log(f"  Cache: {i + 1}/{N}")

    filled = int(mask.sum())
    total  = N * N_SLOT
    log(f"Cache complete: {filled}/{total} slots filled ({filled/max(total,1):.1%})")
    return cache, mask

log("Building shared 6-slot cache...")
cache_6slot, mask_6slot = build_6slot_cache(TEST_IDS, slot_map, lat_map)

# %% [markdown]
# ## Arm 1 — CoAtNet (Raptor)
#
# The Raptor model uses a 5-slot, 64-slice scheme with mm-cropping at 140mm.
# A CoAtNet backbone processes 3-slice windows (2.5D).
# Per-finding attention pools windows into one prediction per study.
# This is inference-only: weights are loaded from `raptor-knee-widedense`.

# %% [code] — CoAtNet (Raptor) model definition
IMG_RAPTOR  = 384
CROP_RAPTOR = 140.0
K_EVAL      = 42    # number of windows evaluated per study

# Slot scheme for Raptor (5 slots, 64 slices total)
RAPTOR_SLOTS = [
    ("Sagittal", 1, 18),   # fluid-sensitive sagittal, 18 slices
    ("Sagittal", 0, 14),   # structural sagittal, 14 slices
    ("Coronal",  1, 12),   # fluid-sensitive coronal, 12 slices
    ("Coronal",  0,  8),   # structural coronal, 8 slices
    ("Axial",   -1, 12),   # axial (any), 12 slices
]
RAPTOR_MAXS = sum(s[2] for s in RAPTOR_SLOTS)  # 64

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class RaptorClassifier(nn.Module):
    """
    CoAtNet-based per-finding attention model.
    Architecture must match the saved weights exactly.
    """
    def __init__(self, backbone, F_dim: int = 768, n: int = 12, drop: float = 0.2):
        super().__init__()
        self.backbone = backbone
        self.norm     = nn.LayerNorm(F_dim)
        self.att      = nn.Sequential(
            nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
            nn.Linear(256, n),
        )
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)          # (B, K, n)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)    # (B, n, F)
        return (pooled * self.clsW).sum(-1) + self.clsb # (B, n)

    def forward(self, x):
        return self.head(self.encode(x))

def _build_raptor_backbone(arch, pretrained=False):
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)

def load_raptor(pt_path, device):
    ck   = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k")
    bb   = _build_raptor_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    del ck
    gc.collect()
    return model

# %% [code] — Raptor preprocessing and inference
def _pick_raptor_series(rows, plane, fluid):
    """Pick the best series for a Raptor slot."""
    cands = [r for r in rows if r.get("Anatomical_Plane") == plane]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None

def _read_raptor_slice(fpath, ps):
    """Decode one DICOM slice and mm-crop to CROP_RAPTOR."""
    ds  = pydicom.dcmread(fpath)
    arr = apply_modality_lut(ds.pixel_array, ds).astype(np.float32)
    if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        arr = arr.max() - arr
    h, w = arr.shape
    want = int(round(CROP_RAPTOR / max(ps, 1e-3)))
    want = min(want, min(h, w))
    y0, x0 = (h - want) // 2, (w - want) // 2
    arr = arr[y0:y0 + want, x0:x0 + want]
    return cv2.resize(arr, (IMG_RAPTOR, IMG_RAPTOR), interpolation=cv2.INTER_AREA)

def build_raptor_volume(study_uid, ser_records):
    """Build the 64-slice volume for Raptor from DICOM files."""
    rows = ser_records.get(str(study_uid), [])
    vol  = np.zeros((RAPTOR_MAXS, IMG_RAPTOR, IMG_RAPTOR), dtype=np.uint8)
    used = set()
    idx  = 0

    for plane, fluid, k in RAPTOR_SLOTS:
        r = _pick_raptor_series(
            [rr for rr in rows if rr.get("SeriesInstanceUID") not in used],
            plane, fluid,
        )
        if r is None:
            idx += k
            continue
        used.add(r["SeriesInstanceUID"])
        sdir  = TEST_SER / str(study_uid) / str(r["SeriesInstanceUID"])
        files_raw = [f.name for f in sdir.glob("*.dcm")]
        try:
            files_ordered = _order_by_geometry(files_raw, str(sdir))
            files = [sdir / f for f in files_ordered]
        except Exception:
            files = sorted(sdir.glob("*.dcm"))
        if not files:
            idx += k
            continue

        n   = len(files)
        lo  = int(n * 0.06)
        hi  = max(int(n * 0.94) - 1, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k

        # Compute series-level percentile
        arrs = []
        for p in picks:
            fp = files[min(p, n - 1)]
            try:
                ds  = pydicom.dcmread(str(fp))
                arr = apply_modality_lut(ds.pixel_array, ds).astype(np.float32)
                ps  = float(getattr(ds, "PixelSpacing", [0.5])[0])
                arrs.append((arr, ps))
            except Exception:
                arrs.append(None)

        valid = [a[0] for a in arrs if a is not None]
        if valid:
            all_px = np.concatenate([a.ravel() for a in valid])
            lo_v, hi_v = np.percentile(all_px, [2, 98])
        else:
            lo_v, hi_v = 0.0, 1.0

        for a_ps in arrs:
            if idx >= RAPTOR_MAXS:
                break
            if a_ps is None:
                idx += 1
                continue
            a, ps = a_ps
            aw = np.clip((a - lo_v) / (hi_v - lo_v + 1e-6), 0, 1)
            h, w = aw.shape
            want = int(round(CROP_RAPTOR / max(ps, 1e-3)))
            want = min(want, min(h, w))
            y0, x0 = (h - want) // 2, (w - want) // 2
            aw = aw[y0:y0 + want, x0:x0 + want]
            aw = cv2.resize(aw, (IMG_RAPTOR, IMG_RAPTOR), interpolation=cv2.INTER_AREA)
            vol[idx] = (aw * 255).astype(np.uint8)
            idx += 1

    mask = (vol.reshape(RAPTOR_MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask

def _eval_windows(vol, mask, k=K_EVAL, res=IMG_RAPTOR):
    """Build 3-slice sliding-window tensors."""
    D    = vol.shape[0]
    if D < 3:
        # Not enough slices for a 3-slice window; pad to 3
        pad = np.zeros((3 - D, res, res), dtype=vol.dtype)
        vol = np.concatenate([vol, pad], axis=0)
        D = 3
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if lo <= c - 1 and c + 1 <= hi]
    if not cs:
        # Fallback: use the middle slice, clamped so c-1 >= 0 and c+1 < D
        c_mid = max(1, min((lo + hi) // 2, D - 2))
        cs = [c_mid]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    centers = [cs[i] for i in idx]

    wins = np.empty((len(centers), 3, res, res), np.float32)
    for j, c in enumerate(centers):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], (res, res), mode="bilinear", align_corners=False)[0]
        wins[j] = t.numpy()

    x = torch.from_numpy(wins)
    x = (x - _IMAGENET_MEAN) / _IMAGENET_STD
    return x

@torch.no_grad()
def raptor_infer_study(model, vol, mask, device):
    xw = _eval_windows(vol, mask)
    x  = xw.unsqueeze(0).to(device)
    try:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(x).float()
    except RuntimeError:
        torch.cuda.empty_cache()
        logits = model(x).float()
    return torch.sigmoid(logits)[0].cpu().numpy()  # (12,)

def run_raptor_arm(test_ids, ser_records_raptor, device, raptor_dir):
    """Run CoAtNet arm. Returns (N, 12) per-finding probabilities."""
    if raptor_dir is None:
        log("Raptor weights not found — skipping arm.")
        return None

    ckpts = sorted(raptor_dir.glob("*.pt"))
    if not ckpts:
        log("No Raptor .pt files found — skipping arm.")
        return None

    if time.time() - T0 > TIME_BUDGET:
        log("Time budget exceeded before Raptor arm. Skipping.")
        return None

    N = len(test_ids)
    fold_preds = []

    for wt_path in ckpts:
        if time.time() - T0 > TIME_BUDGET:
            log("Time budget exceeded mid-Raptor. Skipping remaining Raptor folds.")
            break
            
        log(f"Loading Raptor from {wt_path.name}...")
        model = load_raptor(wt_path, device)
        probs = np.full((N, N_TARGETS), np.nan, dtype=np.float32)

        for i, study in enumerate(test_ids):
            try:
                vol, mask = build_raptor_volume(study, ser_records_raptor)
                probs[i]  = raptor_infer_study(model, vol, mask, device)
            except Exception as e:
                log(f"  Raptor study {i} fallback ({type(e).__name__}: {e})")
            if (i + 1) % 50 == 0 or i + 1 == N:
                log(f"  Raptor: {i + 1}/{N}")
        
        # Degenerate Prediction Safety Net
        if float(np.std(probs)) < 1e-09:
            log(f"  Degenerate predictions for {wt_path.name}; discarded.")
        else:
            fold_preds.append(probs)
            
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if not fold_preds:
        return {}

    log("Raptor arm complete.")
    return {f"Raptor_Fold_{i+1}": p for i, p in enumerate(fold_preds)}

# Build the series record lookup for Raptor
SER_RECORDS_RAPTOR = {
    k: v.to_dict("records")
    for k, v in test_ser.groupby("StudyInstanceUID")
}


# %% [markdown]
# ## Arm 2 — DINOv3 Fold Models
#
# Five fold checkpoints of DINOv3 ViT-S/16, fine-tuned on this competition.
# Each fold checkpoint is a complete network (backbone + slot-conditioning + readout).
# We use fold-level rank aggregation to reduce overfitting.

# %% [code] — DINOv3 fold model definitions
# Declare the slot type count and padding index (must match training)
FOLD_N_SLOT_TYPES = 6
FOLD_MASK_IDX     = 0
FOLD_N_SLICE      = 16

# Architecture classes (simplified versions matching the checkpoint keys)
class _SlotHead(nn.Module):
    """Simple per-finding slot attention head."""
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj     = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query    = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop     = nn.Dropout(p)
        self.out      = nn.Linear(hidden, n_out)
        self.hidden   = hidden

    def forward(self, x, mask):
        # x: (B, S, dim), mask: (B, S)
        h   = self.proj(x) + self.slot_emb                        # (B, S, hidden)
        att = torch.einsum("bsh,oh->bos", h, self.query) / math.sqrt(self.hidden)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class _DINOv2Model(nn.Module):
    """DINOv2 backbone + slot-head for 6-slot inference."""
    def __init__(self, backbone, dim, pool="cls_mean"):
        super().__init__()
        self.backbone = backbone
        self.pool     = pool
        parts = {"cls_mean": 2, "cls_mean_focal": 3}
        self.head = _SlotHead(dim * parts.get(pool, 2), N_SLOT, N_TARGETS)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, (img_size, img_size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        out   = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

def _load_dinov2_model(dino_path, state_dict, pool="cls_mean", device=DEV):
    from transformers import AutoModel
    bb  = AutoModel.from_pretrained(str(dino_path))
    dim = bb.config.hidden_size
    m   = _DINOv2Model(bb, dim, pool=pool)
    m.load_state_dict(state_dict, strict=True)
    return m.eval().to(device)

# %% [code] — DINOv3 fold model — we use the same _DINOv2Model class
# The DINOv3 checkpoints store the full backbone + head, no external DINOv2 needed.
# They are loaded via timm (ViT-S/16 architecture) instead of transformers.

class _DINOv3Head(nn.Module):
    def __init__(self, dim=768, n_slot=6, n_out=12, hidden=256):
        super().__init__()
        # Checkpoint explicitly contains these exact shapes for 768-dim input
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden))
        self.query    = nn.Parameter(torch.randn(n_out, hidden))
        self.proj     = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.out = nn.Linear(hidden, n_out)

    def forward(self, feat, mask):
        # feat: (B, 6, 768), mask: (B, 6)
        B, S, D = feat.shape
        x = self.proj(feat) + self.slot_emb.unsqueeze(0)
        
        q = self.query.unsqueeze(0).expand(B, -1, -1)
        # Scaled dot-product attention (manual, no projection matrices)
        scores = torch.bmm(q, x.transpose(1, 2)) / math.sqrt(x.shape[-1])
        
        # Mask out missing slots
        mask_expanded = mask.unsqueeze(1).expand(-1, q.shape[1], -1)
        scores = scores.masked_fill(mask_expanded == 0, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        attended = torch.bmm(attn, x) # (B, 12, 256)
        
        # Element-wise per-target projection
        # out.weight is [12, 256], out.bias is [12]
        preds = (attended * self.out.weight.unsqueeze(0)).sum(dim=-1) + self.out.bias.unsqueeze(0)
        return preds

class _DINOv3Model(nn.Module):
    """
    DINOv3 ViT-S/16 with slot-type token conditioning.
    Matches the fold checkpoint architecture from knee-mri-fold-weights.
    """
    def __init__(self, backbone, n_slot_types=FOLD_N_SLOT_TYPES, n_out=N_TARGETS, dim=384, img_size=518):
        super().__init__()
        self.img_size  = img_size
        self.backbone  = backbone
        self.head      = _DINOv3Head(dim=dim * 2, n_slot=n_slot_types, n_out=n_out)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        # imgs: (B, N_SLOT, GROUP, H, W)  mask: (B, N_SLOT)
        B, S, G, H, W = imgs.shape
        x   = imgs.reshape(B * S, G, H, W).float().div_(255.0)
        x   = (x - self.mean) / self.std
        # Encode each slot's 3 slices as a single pseudo-RGB image (pick middle slice)
        # In practice these models were trained with a DepthCompress stem — we replicate
        # by taking the middle slice and repeating it 3 times as RGB channels.
        mid = G // 2
        xm  = x[:, [mid, mid, mid]]               # (B*S, 3, H, W)
        
        # Resize to match ViT configured image size (e.g. 518)
        if xm.shape[-1] != self.img_size:
            xm = F.interpolate(xm, (self.img_size, self.img_size), mode="bilinear", align_corners=False)
        
        # HuggingFace ViT forward pass (extract CLS token)
        out = self.backbone(pixel_values=xm)
        f   = out.last_hidden_state[:, 0]          # (B*S, dim)
        patch = out.last_hidden_state[:, 1:]       # (B*S, 1369, dim)
        
        parts = [f, patch.mean(1)]
        feat  = torch.cat(parts, dim=1).reshape(B, S, -1) # (B, 6, 768)
        
        return self.head(feat, mask)

def run_dinov3_arm(test_ids, cache, mask, device, fold_dir):
    """
    Load DINOv3 fold checkpoints and run fold-rank aggregation.
    Returns (N, N_TARGETS) rank-aggregated predictions.
    """
    if fold_dir is None or not fold_dir.is_dir():
        log("DINOv3 fold checkpoints not found — skipping arm.")
        return None

    ckpts = sorted(fold_dir.glob("*.pt"))
    if not ckpts:
        log("No .pt files found in DINOv3 dir — skipping arm.")
        return None

    if time.time() - T0 > TIME_BUDGET:
        log("Time budget exceeded before DINOv3 arm. Skipping.")
        return None

    log(f"Found {len(ckpts)} DINOv3 fold checkpoints.")
    N = len(test_ids)
    fold_preds = []

    for ckpt_path in ckpts:
        if time.time() - T0 > TIME_BUDGET:
            log("Time budget exceeded mid-DINOv3. Skipping remaining folds.")
            break
            
        log(f"  Loading {ckpt_path.name}...")
        z   = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        cfg = z.get("cfg", {})

        backbone_name = cfg.get("backbone", "vit_small_patch16_224.dino")
        n_slice       = cfg.get("n_slice", FOLD_N_SLICE)
        img_size      = cfg.get("img", CACHE_IMG)

        try:
            from transformers import Dinov2Config, Dinov2Model
            
            # The missing 'intermediate.dense' keys prove this is specifically 
            # a Dinov2Model (which uses 'mlp.fc1') rather than a standard ViTModel.
            hf_img_size = 518
            hf_config = Dinov2Config(
                image_size=hf_img_size,
                patch_size=14,
                num_hidden_layers=12,
                hidden_size=384,
                mlp_ratio=4,
                num_attention_heads=6,
            )
            bb = Dinov2Model(hf_config)
            
            model = _DINOv3Model(bb, dim=hf_config.hidden_size, img_size=hf_img_size)
            # Auto-detect checkpoint key format
            if isinstance(z, dict):
                for key in ("state_dict", "model", "model_state_dict"):
                    if key in z:
                        sd = z[key]
                        break
                else:
                    # Assume the dict itself is a raw state_dict
                    sd = z
            else:
                sd = z
            missing_cnt, unexp_cnt = check_backbone_keys(model, sd, model_name=f"DINOv3_{ckpt_path.name}")
        except Exception as e:
            log(f"    Failed to load {ckpt_path.name}: {e}")
            continue

        model.eval().to(device)

        probs = np.full((N, N_TARGETS), np.nan, dtype=np.float32)
        BATCH = 8

        with torch.no_grad():
            for b0 in range(0, N, BATCH):
                b1   = min(b0 + BATCH, N)
                imgs = torch.from_numpy(cache[b0:b1]).to(device)
                msk  = torch.from_numpy(mask[b0:b1]).to(device)
                try:
                    with torch.autocast("cuda", enabled=device.type == "cuda"):
                        out_orig = torch.sigmoid(model(imgs, msk).float())
                        # Test-time augmentation: horizontal flip across spatial W dimension
                        imgs_flip = torch.flip(imgs, dims=[-1])
                        out_flip = torch.sigmoid(model(imgs_flip, msk).float())
                        # Swap medial and lateral compartment target predictions on flipped images
                        med_m_idx, lat_m_idx = TARGETS.index("Medial Meniscus"), TARGETS.index("Lateral Meniscus")
                        med_oa_idx, lat_oa_idx = TARGETS.index("Medial OA"), TARGETS.index("Lateral OA")
                        out_flip_swapped = out_flip.clone()
                        out_flip_swapped[:, med_m_idx] = out_flip[:, lat_m_idx]
                        out_flip_swapped[:, lat_m_idx] = out_flip[:, med_m_idx]
                        out_flip_swapped[:, med_oa_idx] = out_flip[:, lat_oa_idx]
                        out_flip_swapped[:, lat_oa_idx] = out_flip[:, med_oa_idx]
                        out = 0.5 * (out_orig + out_flip_swapped)
                    probs[b0:b1] = out.cpu().numpy()
                except Exception:
                    pass

        # Degenerate Prediction Safety Net
        if float(np.std(probs)) < 1e-09:
            log(f"  Degenerate predictions for {ckpt_path.name}; discarded.")
        else:
            fold_preds.append(probs)
            
        del model, z
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if not fold_preds:
        return {}

    log(f"DINOv3: collected {len(fold_preds)} folds.")
    return {f"DINOv3_Fold_{i+1}": p for i, p in enumerate(fold_preds)}



# %% [markdown]
# ## Arm 3 — RadImageNet (Optional)
#
# ResNet-50 pretrained on medical images. Frozen encoder with trained
# attention heads from the E10 layout (3 planes: SAG, COR, AX fluid-sensitive).
# Added at low weight (0.20) to avoid over-relying on one family.

# %% [code] — RadImageNet arm definition and inference
from torchvision.models import resnet50

class _RadAttnHead(nn.Module):
    """Attention head for RadImageNet encoder features."""
    def __init__(self, feat_dim=2048, n_slot=3, n_out=N_TARGETS, hidden=256):
        super().__init__()
        self.slot_proj = nn.Linear(feat_dim, hidden)
        self.query     = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.norm      = nn.LayerNorm(hidden)
        self.out       = nn.Linear(hidden, n_out)

    def forward(self, feats, mask):
        # feats: (B, S, feat_dim), mask: (B, S)
        h   = self.slot_proj(feats)                           # (B, S, hidden)
        h   = self.norm(h)
        att = torch.einsum("bsh,oh->bos", h, self.query) / math.sqrt(h.shape[-1])
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = torch.einsum("bos,bsh->boh", att, h)           # (B, n_out, hidden)
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

def _build_rad_encoder():
    enc = resnet50(weights=None)
    enc.fc = nn.Identity()
    return enc

def _find_rad_weights(head_dir):
    # RadImageNet ResNet-50 backbone
    rad_enc = None
    
    # 1. Check the explicit marwan folder inside the repro-assets bundle
    marwan_dir = Path("/kaggle/input/rsna-knee-bend-dinov3-0917-repro-assets/resnet-50-radimagenet-marwan")
    if not marwan_dir.exists():
        # Maybe it's mounted under /datasets/tonylica/...
        marwan_dir = Path("/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/resnet-50-radimagenet-marwan")
        
    if marwan_dir.is_dir():
        # Grab any .pth or .pt file inside it (the filename might not be what we expect)
        for ext in ["*.pth", "*.pt"]:
            for p in marwan_dir.glob(ext):
                rad_enc = str(p)
                break
            if rad_enc: break

    # 2. Fast fallback: check common Kaggle paths directly
    if not rad_enc:
        fallbacks = [
            "/kaggle/input/radimagenet-resnet50-notop-torch/RadImageNet-ResNet50_notop_torch.pth",
            "/kaggle/input/radimagenet/pytorch/1/RadImageNet-ResNet50_notop_torch.pth",
        ]
        for f in fallbacks:
            if os.path.isfile(f):
                rad_enc = f
                break

    # E10 attention head files
    heads = sorted(head_dir.glob("*.pt")) if head_dir and head_dir.is_dir() else []
    return rad_enc, heads

def run_radimagenet_arm(test_ids, cache, mask, device, head_dir):
    """Run RadImageNet ResNet-50 + E10 heads. Returns (N, N_TARGETS) rank-aggregated."""
    rad_enc_path, head_paths = _find_rad_weights(head_dir)

    if rad_enc_path is None:
        log("RadImageNet encoder not found — skipping Rad arm.")
        return None
    if not head_paths:
        log("No RadImageNet head checkpoints found — skipping Rad arm.")
        return None

    if time.time() - T0 > TIME_BUDGET:
        log("Time budget exceeded before RadImageNet arm. Skipping.")
        return None

    # ── Build the shared encoder ────────────────────────────────────────────
    log(f"Loading RadImageNet encoder from {rad_enc_path}...")
    enc = _build_rad_encoder()
    try:
        rad_z = torch.load(rad_enc_path, map_location="cpu", weights_only=False)
        if isinstance(rad_z, dict):
            # Extract actual state dict if wrapped
            for k in ("state_dict", "model", "model_state_dict"):
                if k in rad_z:
                    rad_z = rad_z[k]
                    break
        enc.load_state_dict(rad_z, strict=False)
    except Exception as e:
        log(f"  Encoder load failed: {e} — skipping Rad arm.")
        return None
    enc.eval().to(device)

    # ── E10 slots: SAG_FLUID_FS(0), COR_FLUID_FS(1), AX_FLUID_FS(2) ────────
    E10_SLOTS = [0, 1, 2]
    N = len(test_ids)

    # Extract encoder features for the three E10 slots
    # Each slot has GROUP=3 slices; feed all 3 as RGB channels (2.5D)
    log("Extracting RadImageNet features for E10 slots (2.5D channels)...")
    rad_feats = np.zeros((N, len(E10_SLOTS), 2048), dtype=np.float32)
    rad_mask  = np.zeros((N, len(E10_SLOTS)), dtype=np.float32)
    BATCH = 16
    mu = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    sd = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    with torch.no_grad():
        for si, slot_idx in enumerate(E10_SLOTS):
            for b0 in range(0, N, BATCH):
                b1   = min(b0 + BATCH, N)
                # Feed all 3 adjacent slices as R, G, B channels
                # cache shape: (N, 6_slots, GROUP=3, H, W)
                imgs = torch.from_numpy(
                    cache[b0:b1, slot_idx]  # (batch, 3, H, W) — all 3 slices
                ).float().div_(255.0).to(device)
                imgs = (imgs - mu) / sd
                # Resize to 224 for ResNet
                imgs = F.interpolate(imgs, (224, 224), mode="bilinear", align_corners=False)
                f = enc(imgs).float().cpu().numpy()  # (batch, 2048)
                rad_feats[b0:b1, si] = f
                rad_mask[b0:b1, si]  = mask[b0:b1, slot_idx]

    del enc
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # ── Run each head ────────────────────────────────────────────────────────
    fold_preds = []
    for hpath in head_paths:
        if time.time() - T0 > TIME_BUDGET:
            log("Time budget exceeded mid-RadImageNet. Skipping remaining heads.")
            break
            
        try:
            ck   = torch.load(hpath, map_location="cpu", weights_only=False)
            head = _RadAttnHead(feat_dim=2048, n_slot=len(E10_SLOTS))
            sd_head = ck.get("model", ck)
            # Strip possible module/model prefixes from state dict
            sd_clean = {}
            for k, v in sd_head.items():
                k_clean = k.replace("module.", "").replace("model.", "")
                sd_clean[k_clean] = v
            head.load_state_dict(sd_clean, strict=False)
            head.eval().to(device)

            probs = np.full((N, N_TARGETS), np.nan, dtype=np.float32)
            BATCH = 32
            with torch.no_grad():
                for b0 in range(0, N, BATCH):
                    b1   = min(b0 + BATCH, N)
                    feats_t = torch.from_numpy(rad_feats[b0:b1]).to(device)
                    mask_t  = torch.from_numpy(rad_mask[b0:b1]).to(device)
                    out = torch.sigmoid(head(feats_t, mask_t).float())
                    probs[b0:b1] = out.cpu().numpy()

            if float(np.std(probs)) < 1e-09:
                log(f"  Degenerate predictions for Rad head {hpath.name}; discarded.")
            else:
                fold_preds.append(probs)
                
            del head, ck
            gc.collect()
        except Exception as e:
            log(f"  RadImageNet head {hpath.name} failed: {e}")

    if not fold_preds:
        return {}

    log(f"RadImageNet: collected {len(fold_preds)} heads.")
    return {f"RadImageNet_Head_{i+1}": p for i, p in enumerate(fold_preds)}



# %% [markdown]
# ## Final Ensemble with Metadata-Conditioned Blending
#
# ### Base blend weights (design choices, not tuned on 58 gold labels):
# - CoAtNet: 0.40 — strong single model, diverse architecture (CNN+Transformer)
# - DINOv3:  0.45 — 5 folds, more robust aggregation
# - RadImageNet: 0.15 — adds medical pretraining signal
#
# ### Per-finding overrides from notebook analysis:
# - Baker's and Fracture: RadImageNet weight = 0 (it hurts these two)
# - Focal findings: max pooling was already applied at the arm level
#
# ### Metadata-conditioned modulation:
# - Each arm's weight per (study, finding) is scaled by slot coverage
# - Studies with full slot coverage are fully trusted
# - Studies with missing slots get reduced confidence on the findings that depend on them

# %% [code] — Final ensemble
def build_final_ensemble(
    member_preds_dict,
    member_coverages_dict,
    test_ids,
    base_family_weights={"raptor": 0.40, "dinov3": 0.45, "radimagenet": 0.15},
):
    N = len(test_ids)
    log("Building multi-representation dynamic member-level ensemble...")

    arms, base_weights, arm_names, arm_families, arm_coverages = [], [], [], [], []

    family_members = {"raptor": [], "dinov3": [], "radimagenet": []}
    for m_name, preds in member_preds_dict.items():
        if preds is None: continue
        fam = "raptor" if "raptor" in m_name.lower() or "coatnet" in m_name.lower() else ("dinov3" if "dino" in m_name.lower() else "radimagenet")
        family_members[fam].append((m_name, preds, member_coverages_dict.get(m_name)))

    for fam, mems in family_members.items():
        if not mems: continue
        fam_w = base_family_weights.get(fam, 0.33)
        per_mem_w = fam_w / len(mems)
        for m_name, preds, cov in mems:
            p_clean = preds.copy()
            if fam == "radimagenet":
                for t in ["Baker's", "Fracture"]:
                    ti = TARGETS.index(t)
                    p_clean[:, ti] = np.nan
            arms.append(p_clean)
            base_weights.append(per_mem_w)
            arm_names.append(m_name)
            arm_families.append(fam)
            arm_coverages.append(cov if cov is not None else np.ones((N, N_TARGETS), dtype=np.float64))

    if not arms:
        log("No active ensemble members available! Defaulting to 0.5.")
        return np.full((N, N_TARGETS), 0.5, dtype=np.float32)

    log(f"Active members ({len(arms)}): {arm_names}")

    member_dict = {name: arr for name, arr in zip(arm_names, arms)}
    log_pairwise_spearman(member_dict, TARGETS)

    final = metadata_conditioned_blend(
        arm_preds=arms,
        base_weights=base_weights,
        arm_coverages=arm_coverages,
        alpha=0.4,
        arm_families=arm_families,
    )

    stacked_raw = np.array([np.nan_to_num(a, nan=0.0) for a in arms])
    raw_max = stacked_raw.max(axis=0)
    focal_cols = [TARGETS.index(t) for t in FOCAL_FINDINGS if t in TARGETS]
    for c in focal_cols:
        focal_boost = np.maximum(0.0, raw_max[:, c] - 0.80) * 0.20
        final[:, c] = final[:, c] + focal_boost

    train_gt_df = pd.read_csv(TRAIN_CSV) if TRAIN_CSV.exists() else None
    final = apply_hierarchical_consistency(final, TARGETS, train_gt_df=train_gt_df)

    final = rank_pct(final)

    log(f"Ensemble complete. Shape: {final.shape}")
    return final

raptor_members = run_raptor_arm(TEST_IDS, SER_RECORDS_RAPTOR, DEV, RAPTOR_DIR) if 'run_raptor_arm' in globals() else {}
dinov3_members = run_dinov3_arm(TEST_IDS, cache_6slot, mask_6slot, DEV, DINO_FOLD_DIR) if 'run_dinov3_arm' in globals() else {}
rad_members    = run_radimagenet_arm(TEST_IDS, cache_6slot, mask_6slot, DEV, RAD_HEADS_DIR) if 'run_radimagenet_arm' in globals() else {}

all_members = {}
if isinstance(raptor_members, dict): all_members.update(raptor_members)
if isinstance(dinov3_members, dict): all_members.update(dinov3_members)
if isinstance(rad_members, dict):    all_members.update(rad_members)

member_coverages = {
    m: (finding_coverage if "rad" in m.lower() or "dino" in m.lower() else np.ones((len(TEST_IDS), N_TARGETS), dtype=np.float32))
    for m in all_members.keys()
}

final_preds = build_final_ensemble(
    member_preds_dict=all_members,
    member_coverages_dict=member_coverages,
    test_ids=TEST_IDS,
    base_family_weights={"raptor": 0.40, "dinov3": 0.45, "radimagenet": 0.15},
)

# %% [markdown]
# ## Write Submission
#
# The final predictions are already percentile ranks in [0, 1].
# We write them directly to submission.csv aligned to the sample submission order.

# %% [code] — Write submission
def write_submission(preds, test_ids, sample_df, out_path):
    """Write a submission CSV aligned to the sample submission row order."""
    # ── Derive the actual target column names from sample_submission.csv ────
    sub_cols = [c for c in sample_df.columns if c != "StudyInstanceUID"]
    log(f"Sample submission columns: {sub_cols}")

    if len(sub_cols) != N_TARGETS:
        log(f"WARNING: Expected {N_TARGETS} target columns but sample_submission has {len(sub_cols)}. "
            f"Will use sample_submission column names directly.")

    # Build a mapping: our internal TARGETS index → sample_submission column name
    # Strategy: fuzzy match by lowercasing and stripping punctuation
    import re as _re
    def _norm(s): return _re.sub(r"[^a-z0-9]", "", s.lower())
    our_norm   = {_norm(t): (i, t) for i, t in enumerate(TARGETS)}
    sub_norm   = {_norm(c): c for c in sub_cols}

    # Try exact normalised match first
    col_order  = []   # list of (our_index, sub_col_name)
    unmatched  = []
    for sc in sub_cols:
        key = _norm(sc)
        if key in our_norm:
            col_order.append((our_norm[key][0], sc))
        else:
            unmatched.append(sc)

    if unmatched:
        log(f"WARNING: Could not match these submission columns to TARGETS: {unmatched}")
        log(f"  Filling them with 0.5 (neutral prediction).")

    # ── Build the submission DataFrame ─────────────────────────────────────
    sub = pd.DataFrame({"StudyInstanceUID": test_ids})
    for our_idx, sub_col in col_order:
        sub[sub_col] = preds[:, our_idx]
    for um in unmatched:
        sub[um] = 0.5

    # Reorder columns to match sample_submission exactly
    sub = sub[["StudyInstanceUID"] + sub_cols]

    # ── Align rows to sample submission order ──────────────────────────────
    sub = sample_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[sub_cols] = sub[sub_cols].fillna(0.5)

    # ── Sanity checks ──────────────────────────────────────────────────────
    assert list(sub.columns) == list(sample_df.columns), \
        f"Column order mismatch!\n  Expected: {list(sample_df.columns)}\n  Got:      {list(sub.columns)}"
    assert len(sub) == len(sample_df), \
        f"Row count mismatch! Expected {len(sample_df)}, got {len(sub)}"
    assert np.isfinite(sub[sub_cols].values).all(), "NaN/Inf in predictions"

    # ── Write ──────────────────────────────────────────────────────────────
    sub.to_csv(out_path, index=False)
    log(f"Wrote {out_path} | shape={sub.shape}")
    log(f"  Columns: {list(sub.columns)}")
    log(f"  Value range: [{sub[sub_cols].values.min():.4f}, {sub[sub_cols].values.max():.4f}]")
    log(f"  Mean prediction per column:")
    for c in sub_cols:
        log(f"    {c:25s} → mean={sub[c].mean():.4f}  std={sub[c].std():.4f}")
    return sub

sub = write_submission(final_preds, TEST_IDS, sample, WORKING / "submission.csv")
print("\n── Submission preview (first 5 rows) ──")
print(sub.head().to_string(index=False))

# %% [markdown]
# ## Diagnostic: Coverage Distribution
#
# Shows which findings have low slot coverage in the test set.
# Low-coverage findings are candidates for the metadata-conditioned
# blend to reduce confidence in those specific arm predictions.

# %% [code] — Coverage diagnostics
print("\n── Finding Coverage (mean across test studies) ──")
cov_df = pd.DataFrame({
    "Finding":  TARGETS,
    "Mean Coverage": finding_coverage.mean(0).round(3),
    "Min Coverage":  finding_coverage.min(0).round(3),
    "Pct Below 0.5": (finding_coverage < 0.5).mean(0).round(3),
}).sort_values("Mean Coverage")
print(cov_df.to_string(index=False))

print("\n── Slot fill rate per slot ──")
slot_df = pd.DataFrame({
    "Slot":      SLOT_NAMES,
    "Fill Rate": slot_presence.mean(0).round(3),
})
print(slot_df.to_string(index=False))

print("\n── Active Member & Ensemble prediction ranges ──")
for name, probs in list(all_members.items()) + [("Final Ensemble", final_preds)]:
    if probs is not None:
        p_valid = probs[~np.isnan(probs)]
        if len(p_valid) > 0:
            print(f"{name:30s}  mean={p_valid.mean():.3f}  std={p_valid.std():.3f}  "
                  f"min={p_valid.min():.3f}  max={p_valid.max():.3f}")
        else:
            print(f"{name:30s}  [All NaN / missing]")

print("\nDone!")

cuda:0 OK — Tesla T4 sm_75
cuda:1 OK — Tesla T4 sm_75
Primary device: cuda:0
[    1.7s] Competition  : /kaggle/input/competitions/rsna-knee-abnormality-detection
[    1.7s] DINO Folds   : /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-weights
[    1.7s] Rad Heads    : /kaggle/input/datasets/mattiaangeli/rsna-knee-radimagenet-foldsv1-heads
[    1.7s] Raptor dir   : /kaggle/input/datasets/dreaddevelopment/raptor-knee-widedense
[    1.8s] Test studies: 3 | Test series: 15
[    1.8s] Reading test series headers...
[    1.8s] Probing 15 series in test_series ...
[    1.9s] Headers: 15 series annotated
[    1.9s] Laterality: 1 from tags, 2 from geometry, 0 unknown; disagreed on 0
[    3.3s] Slot map: 3/3 studies have at least one slot
[    3.3s] Slot duplication audit: 0/3 studies (0.0%) have duplicate series assigned to multiple slots.
[    3.3s] Coverage: mean slot fill 0.72, mean finding coverage 0.73
[    3.4s] Building shared 6-slot cache...
[    3.4s]